In [23]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
import torch
from PIL import Image
from transformers import ViTImageProcessor, ViTForImageClassification
from transformers import AutoImageProcessor, AutoModel
from transformers import AutoImageProcessor, ViTForMaskedImageModeling
from transformers import ViTImageProcessor, ViTModel
#from transformers import CLIPProcessor, CLIPModel
import pandas as pd
import numpy as np
import os
import pickle
import glob
import requests


## Embedding

In [2]:
path = "D:\\Research"

In [3]:
artnet_2024 = pd.read_excel(f"{path}\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_Europe_2024_merged.xlsx")

In [4]:
artnet_2024.shape

(355551, 16)

In [5]:
#number_size = int(sys.argv[1])
#range_start = int(sys.argv[2])
number_size = 100000
range_start = 0
range_end = min(range_start+number_size,artnet_2024.shape[0])
N = min(number_size, artnet_2024.shape[0]-range_start)

In [6]:
url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(requests.get(url, stream=True).raw)

In [13]:
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")
model = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")

In [14]:
inputs = processor(images=image, return_tensors="pt")
outputs = model(**inputs)
embedding = outputs.last_hidden_state[:, 0, :]

In [17]:
embedding/ embedding.norm(p=2, dim=-1, keepdim=True)

tensor([[ 3.2173e-02,  1.8872e-02,  3.1328e-02, -4.2783e-03, -1.1009e-02,
         -3.2313e-02,  1.2582e-02, -1.9958e-02,  8.8835e-03, -3.3620e-02,
          3.6828e-02,  2.5923e-02,  5.1362e-02, -3.4238e-02, -2.4275e-02,
          5.2016e-02,  1.1989e-02, -4.3269e-02,  5.4681e-02,  4.0605e-03,
         -6.7684e-02,  7.8017e-04,  1.3085e-02,  4.3195e-02, -3.8757e-02,
          3.8047e-02,  3.5072e-02,  2.2427e-02,  5.4255e-02,  1.4558e-02,
          1.8388e-02,  1.9111e-02, -7.4181e-03,  5.4610e-02, -2.8037e-02,
          3.9386e-02, -1.9974e-02, -1.9547e-02,  1.0279e-02,  2.0046e-02,
          2.6606e-02, -3.8139e-02,  6.5414e-03, -3.0465e-02,  2.0620e-03,
          2.2379e-02, -7.0106e-03, -9.2740e-03,  1.6711e-03, -4.9237e-02,
          2.6217e-02, -2.5630e-02,  1.3407e-02, -1.6497e-02, -4.8966e-02,
         -2.8395e-02, -2.7728e-02,  2.2370e-02, -2.4105e-02, -3.8381e-02,
          4.5637e-02,  4.0651e-02,  2.5990e-02, -1.1253e-02,  2.0795e-03,
         -1.8418e-02,  3.9489e-02, -1.

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [20]:
def convert_one_row(model,i, image_file,device):
    try:
        image = Image.open(f"{path}\\Creativity_Artnet\\Datasets\\ArtNet\\Images\\{image_file}")
        inputs = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            image_features = model(**inputs).last_hidden_state[:, 0, :]
        image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
        image_features = image_features.cpu().numpy()
    except Exception as e:
        print(f"Error processing {image_file}: {e}")
        image_features = np.zeros([1,512])
        
    return i, image_features

In [ ]:
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
embeddings = np.zeros(N, dtype=object)
with ThreadPoolExecutor(max_workers=32) as ex:
    futures = {
        ex.submit(convert_one_row, model,i, f'{artnet_2024.iloc[i]["artwork id"]}.jpg',device): i
        for i in range(range_start,range_end)
    }
    for fut in as_completed(futures):
        i, image_features = fut.result()
        embeddings[i-range_start] =image_features
        if i % 10000 == 0:
            print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: {i}")
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

In [ ]:
np.save(f"Result/clip_embeddings_{range_start}.npy", embeddings)

In [21]:
number_size = 100000
range_start = 0
range_end = min(range_start+number_size,artnet_2024.shape[0])
N = min(number_size, artnet_2024.shape[0]-range_start)
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
while range_start < artnet_2024.shape[0]:
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Now at {range_start}")
    embeddings = np.zeros(N, dtype=object)
    with ThreadPoolExecutor(max_workers=32) as ex:
        futures = {
            ex.submit(convert_one_row, model,i, f'{artnet_2024.iloc[i]["artwork id"]}.jpg',device): i
            for i in range(range_start,range_end)
        }
        for fut in as_completed(futures):
            i, image_features = fut.result()
            embeddings[i-range_start] =image_features
            if i % 10000 == 0:
                print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: {i}")
    np.save(f"{path}\\Creativity_Artnet\\Datasets\\ViT_Embedding_2024\\ViT_embeddings_{range_start}.npy", embeddings)
    range_start = range_start + N
    range_end = min(range_start+number_size,artnet_2024.shape[0])
    N = min(number_size, artnet_2024.shape[0]-range_start)
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Saving embeddings")
    
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

2026-01-01 22:19:34: Start
2026-01-01 22:19:34: Now at 0
2026-01-01 22:19:43: 0
2026-01-01 22:23:18: 10000


The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.


Error processing 424340448.jpg: mean must have 1 elements if it is an iterable, got 3
2026-01-01 22:27:00: 20000
2026-01-01 22:30:42: 30000
2026-01-01 22:34:28: 40000
2026-01-01 22:38:15: 50000
2026-01-01 22:42:01: 60000
2026-01-01 22:45:48: 70000
2026-01-01 22:49:35: 80000
2026-01-01 22:53:19: 90000
2026-01-01 22:57:08: Saving embeddings
2026-01-01 22:57:08: Now at 100000
2026-01-01 22:57:34: 100000
2026-01-01 23:01:15: 110000
2026-01-01 23:05:10: 120000
2026-01-01 23:09:03: 130000
2026-01-01 23:12:53: 140000
2026-01-01 23:16:50: 150000
2026-01-01 23:20:41: 160000
2026-01-01 23:24:19: 170000
2026-01-01 23:28:06: 180000
2026-01-01 23:31:51: 190000
2026-01-01 23:35:38: Saving embeddings
2026-01-01 23:35:38: Now at 200000
2026-01-01 23:36:01: 200000
2026-01-01 23:39:40: 210000
2026-01-01 23:43:22: 220000
2026-01-01 23:47:00: 230000
2026-01-01 23:50:37: 240000
2026-01-01 23:54:14: 250000
2026-01-01 23:57:50: 260000
2026-01-02 00:01:25: 270000
2026-01-02 00:05:02: 280000


The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.


Error processing 440635219.jpg: mean must have 1 elements if it is an iterable, got 3
2026-01-02 00:08:38: 290000
2026-01-02 00:12:15: Saving embeddings
2026-01-02 00:12:15: Now at 300000
2026-01-02 00:12:28: 300000
2026-01-02 00:16:02: 310000
2026-01-02 00:19:38: 320000
2026-01-02 00:23:12: 330000
2026-01-02 00:26:44: 340000
2026-01-02 00:30:20: 350000


The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). A

Error processing 444379706.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444379735.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444379736.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444379767.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444379754.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444379799.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444379818.jpg: mean must have 1 elements if it is an iterable, got 3


The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). A

Error processing 444383507.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444383511.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444383525.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444383531.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444383529.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444383536.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444383532.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444383542.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444383543.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444383568.jpg: mean must have 1 elements if it is an iterable, got 3
Error processing 444383561.jpg: mean must have 1 elements if it is an iterable, got 3
2026-01-02 00:32:20: Saving embeddings
2026-01-02 00:3

In [10]:
artnet_2025.shape

(371778, 26)

## Merge all Outputs

In [24]:
folder_path = f"{path}\\Creativity_Artnet\\Datasets\\ViT_Embedding_2024"
npy_files = glob.glob(f"{folder_path}/*.npy")

In [25]:
arrays = [np.load(f, allow_pickle=True) for f in npy_files]
big_array = np.concatenate(arrays, axis=0)
np.save(f"{path}\\Creativity_Artnet\\Datasets\\ViT_Embedding_2024.npy", big_array)

In [45]:
def fix_embedding(e):
    # Ensure numpy array
    e = np.asarray(e)

    for row in range(e.shape[0]):
    # Case: (1, 512) and all zeros
        if e[row].shape == (1, 512) and np.all(e[row] == 0):
            e[row]=np.zeros((1, 768), dtype=e.dtype)

    # Otherwise leave untouched
    return e

In [46]:
a = fix_embedding(big_array)

In [54]:
a[0]

array([-0.02869199588894844, 0.016623415052890778, -0.012874623760581017,
       -0.032472945749759674, 0.006615957245230675, 0.0003952074912376702,
       0.015522003173828125, 0.009535851888358593, -0.03180920332670212,
       -0.00012342349509708583, 0.03580997884273529, 0.01075754500925541,
       -0.05142274498939514, 0.001748459180817008, 0.02689361572265625,
       -0.01950334571301937, -0.03714055195450783, 0.023027116432785988,
       -0.07364240288734436, -0.019002672284841537, -0.03410565108060837,
       -0.04239894449710846, -0.018800169229507446, -0.03655359521508217,
       0.011225198395550251, -0.01751706376671791, 0.017228348180651665,
       -0.033660467714071274, 0.012066546827554703, 0.029572956264019012,
       -0.019624387845396996, 0.02307063899934292, 0.045335978269577026,
       -0.02341926284134388, 0.043433330953121185, -0.024836450815200806,
       0.05775151774287224, 0.0026827394030988216, 0.0283819530159235,
       -0.07057248800992966, 0.028372343629598

In [48]:
a= np.vstack(a)

In [51]:
np.save(f"{path}\\Creativity_Artnet\\Datasets\\ViT_Embedding_2024.npy", a)

# Formal 2024

In [4]:
artnet_2024 = pd.read_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_Europe_2024_merged.xlsx")

In [5]:
artnet_2024.shape

(355551, 16)

In [ ]:
number_size = 100000
range_start = 0
range_end = min(range_start+number_size,artnet_2025.shape[0])
N = min(number_size, artnet_2025.shape[0]-range_start)

In [6]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32",use_fast=True)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [8]:
def convert_one_row(model,i, image_file,device):
    try:
        image = Image.open(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Images\\{image_file}")
        inputs = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            image_features = model.get_image_features(**inputs)
        image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
        image_features = image_features.cpu().numpy()
    except Exception as e:
        print(f"Error processing {image_file}: {e}")
        image_features = np.zeros([1,512])
        
    return i, image_features

In [ ]:
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
embeddings = np.zeros(N, dtype=object)
with ThreadPoolExecutor(max_workers=32) as ex:
    futures = {
        ex.submit(convert_one_row, model,i, f'{artnet_2025.iloc[i]["artwork id"]}.jpg',device): i
        for i in range(range_start,range_end)
    }
    for fut in as_completed(futures):
        i, image_features = fut.result()
        embeddings[i-range_start] =image_features
        if i % 10000 == 0:
            print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: {i}")
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

In [ ]:
np.save(f"Result/clip_embeddings_{range_start}.npy", embeddings)

In [10]:
number_size = 100000
range_start = 0
range_end = min(range_start+number_size,artnet_2024.shape[0])
N = min(number_size, artnet_2024.shape[0]-range_start)
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
while range_start < artnet_2024.shape[0]:
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Now at {range_start}")
    embeddings = np.zeros(N, dtype=object)
    with ThreadPoolExecutor(max_workers=32) as ex:
        futures = {
            ex.submit(convert_one_row, model,i, f'{artnet_2024.iloc[i]["artwork id"]}.jpg',device): i
            for i in range(range_start,range_end)
        }
        for fut in as_completed(futures):
            i, image_features = fut.result()
            embeddings[i-range_start] =image_features
            if i % 10000 == 0:
                print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: {i}")
    np.save(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\Embedding_2024\\clip_embeddings_{range_start}.npy", embeddings)
    range_start = range_start + N
    range_end = min(range_start+number_size,artnet_2024.shape[0])
    N = min(number_size, artnet_2024.shape[0]-range_start)
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Saving embeddings")
    
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

2025-12-08 03:21:23: Start
2025-12-08 03:21:23: Now at 0
2025-12-08 03:21:34: 0
2025-12-08 03:22:55: 10000


The channel dimension is ambiguous. Got image shape torch.Size([3, 1, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.


2025-12-08 03:24:17: 20000
2025-12-08 03:25:38: 30000
2025-12-08 03:26:59: 40000
2025-12-08 03:28:19: 50000
2025-12-08 03:29:41: 60000
2025-12-08 03:31:02: 70000
2025-12-08 03:32:22: 80000
2025-12-08 03:33:43: 90000
2025-12-08 03:35:05: Saving embeddings
2025-12-08 03:35:05: Now at 100000
2025-12-08 03:35:52: 100000
2025-12-08 03:36:39: 110000
2025-12-08 03:38:01: 120000
2025-12-08 03:39:24: 130000
2025-12-08 03:40:47: 140000
2025-12-08 03:42:10: 150000
2025-12-08 03:43:33: 160000
2025-12-08 03:44:55: 170000
2025-12-08 03:46:18: 180000
2025-12-08 03:47:40: 190000
2025-12-08 03:49:03: Saving embeddings
2025-12-08 03:49:03: Now at 200000
2025-12-08 03:49:40: 200000
2025-12-08 03:50:38: 210000
2025-12-08 03:52:01: 220000
2025-12-08 03:53:23: 230000
2025-12-08 03:54:46: 240000
2025-12-08 03:56:08: 250000
2025-12-08 03:57:31: 260000
2025-12-08 03:58:53: 270000
2025-12-08 04:00:16: 280000


The channel dimension is ambiguous. Got image shape torch.Size([3, 1, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.


2025-12-08 04:01:38: 290000
2025-12-08 04:03:02: Saving embeddings
2025-12-08 04:03:02: Now at 300000
2025-12-08 04:03:41: 300000
2025-12-08 04:04:31: 310000
2025-12-08 04:05:53: 320000
2025-12-08 04:07:16: 330000
2025-12-08 04:08:39: 340000
2025-12-08 04:10:01: 350000


The channel dimension is ambiguous. Got image shape torch.Size([3, 1, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape torch.Size([3, 1, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape torch.Size([3, 1, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is amb

2025-12-08 04:10:47: Saving embeddings
2025-12-08 04:10:47: Ends


# Merging

In [11]:
folder_path = "D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\Embedding_2024"
npy_files = glob.glob(f"{folder_path}/*.npy")

In [12]:
arrays = [np.load(f, allow_pickle=True) for f in npy_files]
big_array = np.concatenate(arrays, axis=0)
np.save("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\clip_embedding_2024.npy", big_array)